# 닥터그린 딸기 병해 — AI Hub 데이터 준비 노트북 (Colab Pro)

이 노트북은 **AI Hub "시설작물(딸기) 개체 이미지 및 시설작물(딸기) 질병 이미지"**(웹 dataSetSn=71451)를
**클래스별로 하나씩** 내려받아, **YOLO 포맷으로 변환 → 클래스당 1,000장 균형 샘플링 → train/val/test 8:1:1 분할
→ data.yaml 생성**까지 자동으로 처리합니다. 이 노트북의 **출력 폴더(OUT_DIR)** 는 학습 노트북
`doctorgreen_yolo_map_boost.ipynb` 의 **입력(DATASET_DIR)** 으로 그대로 들어갑니다.

<b>왜 클래스별로 하나씩 받나요?</b> 5개 질병 클래스 중 "정상" 원천만 75.97GB이고 전체를 합치면 약 194GB라,
한꺼번에 받으면 Colab 디스크가 넘칩니다. 그래서 **"클래스 1개 다운로드 → 변환 → 1,000장 샘플 → 원본 삭제"를
클래스 수만큼 반복**해서, 한 번에 디스크에 남는 원본 용량을 **가장 큰 zip 1개(최대 76GB) 수준**으로 제한합니다.

## 전체 흐름
```
① 클래스별 순차 처리(5회 반복): 다운로드(ts+tl만) → YOLO 변환 → 1,000장 샘플
        → ACCUM_DIR(Drive)에 영구 저장 → RAW_DIR(원본) 삭제
② ACCUM_DIR의 누적 샘플로 sampled 리스트 재구성  →  ③ 층화 8:1:1 분할
        →  ④ data.yaml 생성·검증(박스 시각화)  →  ⑤ 학습 노트북으로 연결
```

## 사전 준비 (반드시 먼저)
1. **AI Hub 내국인 회원가입 + 로그인** (aihub.or.kr). 해외 계정/비로그인은 대용량 다운로드가 제한됩니다.
2. 위 데이터셋 페이지에서 **데이터 활용 신청 → 승인**. **승인은 수동 심사라 수 시간~수일이 걸릴 수 있습니다.**
   승인이 나야 aihubshell 다운로드가 동작합니다.
3. 마이페이지에서 **API Key** 발급(이 노트북에서 `getpass`로 입력, 화면·로그·Drive에 저장하지 않습니다).

<div style="border:2px solid #d33; background:#fff3f3; padding:12px; border-radius:8px">
<b>⚠️ 클래스 이름·순서 경고 (앱이 깨지는 부분)</b><br>
닥터그린 앱/Space(<code>app.py</code>)는 배포된 모델의 <code>model.names</code> 순서로 응답 스키마(disease_name 등)를 만듭니다.
따라서 이 노트북 <b>CONFIG의 <code>CLASS_NAMES</code> 순서는 배포 모델의 <code>model.names</code>와 정확히 같아야</b> 합니다.
학습 노트북을 열어 <b>2번 셀(기존 모델 가져오기)을 실행</b>하면 출력되는 <code>baseline_names</code>(= model.names)를 확인하고,
그 순서를 아래 <code>CLASS_NAMES</code>에 그대로 넣으세요. 순서가 어긋난 채로 학습·배포하면 진단 라벨이 뒤바뀝니다.
</div>


## 0. CONFIG — 여기만 고치세요

아래 값만 상황에 맞게 수정하고, 나머지 셀은 위에서 아래로 순서대로 실행하면 됩니다.
`DATASET_KEY`(기본값 `'305'`)와 클래스별 `FILE_KEYS_BY_CLASS`는 AI Hub 웹의 "딸기" 데이터셋 페이지 →
**파일 목록(API 다운로드)** 탭에서 확인한 실제 값이 이미 채워져 있습니다. 다른 값으로 바뀌었다면
2번 섹션의 `aihubshell -mode l`로 재확인해 갱신하세요.

In [ ]:
# ============ 사용자 설정 (여기만 수정) ============

# aihubshell datasetkey — AI Hub 웹 "딸기" 데이터셋 페이지 > 파일 목록(API 다운로드) 탭에서 확인.
# (2번 섹션의 `-mode l | grep 딸기` 로도 재확인 가능. 웹 dataSetSn 71451과 다를 수 있음)
DATASET_KEY = '71451'

# 클래스별 filekey(원천=ts / 라벨=tl) — AI Hub 웹의 "파일 목록(API 다운로드)" 탭에서 확인한 값.
# ★ 이 딕셔너리의 키는 아래 CLASS_NAMES 항목과 문자열이 정확히 같아야 합니다(순서는 달라도 무방).
# Validation(VS/VL) 파일키는 받지 않습니다 — 이 노트북이 sampled 데이터를 자체적으로 8:1:1 재분할하므로 불필요합니다.
# "정상" 원천(TS)만 75.97GB로 나머지 4개 클래스를 합친 것보다 커서, 아래 3번 섹션에서
# 클래스 하나씩만 내려받아 바로 변환·샘플링한 뒤 원본을 지우는 방식으로 처리합니다.
# 다른 datasetkey/filekey를 쓰려면(예: 데이터셋이 개편된 경우) 이 표만 갱신하면 됩니다.
FILE_KEYS_BY_CLASS = {
    '정상':     {'ts': '475380', 'tl': '475385'},  # 원천 75.97GB
    '역병':     {'ts': '475381', 'tl': '475386'},  # 원천 25.54GB
    '시들음병': {'ts': '475382', 'tl': '475387'},  # 원천 47.24GB
    '잎끝마름': {'ts': '475383', 'tl': '475388'},  # 원천 24.90GB
    '황화':     {'ts': '475384', 'tl': '475389'},  # 원천 20.17GB
}

# 위 filekey들의 대략적인 원천(ts) 용량(GB) — 다운로드 전 디스크 여유공간 안전장치 계산에만 사용됩니다.
# 실제 값과 다소 달라도 동작에는 지장 없습니다(여유공간 부족 시 경고/스킵 판단용 참고치).
CLASS_TS_SIZE_GB = {
    '정상': 75.97, '역병': 25.54, '시들음병': 47.24, '잎끝마름': 24.90, '황화': 20.17,
}

PER_CLASS = 1000                 # 클래스당 목표 샘플 수 (부족하면 있는 만큼)
SPLIT     = (0.8, 0.1, 0.1)      # train / val / test 비율
SEED      = 42                   # 재현용 고정 시드

RAW_DIR = '/content/aihub_raw'   # 다운로드 임시 폴더 — 클래스 처리마다 비웠다가 다시 씀(세션 종료 시 사라짐)
# 클래스별로 변환·샘플링까지 끝난 결과를 누적 저장하는 영구 위치(Drive).
# 세션이 끊겨도 보존되고, 노트북을 다시 실행하면 이미 확보된 클래스는 다운로드를 건너뛰고 이어서 진행합니다.
# 클래스당 최대 PER_CLASS장만 쌓이므로 최종 용량은 원본 대비 매우 작습니다.
ACCUM_DIR = '/content/drive/MyDrive/doctor_green_training/_accum'
# 최종 출력 — Drive에 영구 보존. 학습 노트북의 DATASET_DIR로 이 경로를 그대로 사용합니다.
OUT_DIR = '/content/drive/MyDrive/doctor_green_training/dataset'

# 클래스 이름/순서 — ★배포 모델 model.names 순서에 맞추세요(맨 위 경고 참고).
# 영문 병명으로 배포된 모델이면 여기를 영문으로 바꾸고, 아래 KOR_TO_MODEL로 한글→영문 매핑을 채웁니다.
# (바꿀 경우 위 FILE_KEYS_BY_CLASS / CLASS_TS_SIZE_GB의 키도 CLASS_NAMES와 맞춰 함께 바꾸세요.)
CLASS_NAMES = ['정상', '역병', '시들음병', '잎끝마름', '황화']

# 원천 JSON/폴더의 클래스 표기 → CLASS_NAMES 항목으로의 매핑(필요할 때만).
# 예) 영문 모델: {'정상':'normal','역병':'blight','시들음병':'wilt','잎끝마름':'leaf_tip_burn','황화':'chlorosis'}
# 예) 코드값이면: {'D1':'역병', ...}. 비워두면 원천 표기를 그대로 CLASS_NAMES에서 찾습니다.
KOR_TO_MODEL = {}

# 박스가 없는 이미지(정상 등)를 빈 라벨(배경 이미지)로 포함할지 여부
ALLOW_BACKGROUND = True

# 개체ID(누수 방지 그룹) 정규식. 파일명에서 개체 식별자를 뽑아 같은 개체가 여러 split에 안 가게 함.
# 비우면 파일명 끝의 프레임/일련번호(_0001 등)를 떼어 자동 추정하고, 추정도 애매하면 이미지 단위로 분할(경고 출력).
GROUP_ID_REGEX = ''

# ---- AI Hub JSON 키 매핑 (데이터셋마다 다름 → 3번 섹션에서 처음 다운로드된 클래스의 샘플 JSON을 본 뒤 조정) ----
AIHUB_JSON_KEYS = {
    'annotations': 'annotations',  # 객체(병반) 목록 키
    'bbox':        'bbox',         # 바운딩박스 키 (객체 안)
    'class':       'disease',      # 질병(클래스) 이름 키 (객체 안 또는 최상위)
    'img_width':   'width',        # 이미지 가로 크기 키 (없으면 이미지 파일에서 읽음)
    'img_height':  'height',       # 이미지 세로 크기 키
}
# bbox 값 형태: 'xywh'([x,y,w,h] 픽셀·좌상단기준) | 'xyxy'([x1,y1,x2,y2]) | 'xyxy_dict'({'xtl','ytl','xbr','ybr'})
AIHUB_BBOX_ORDER = 'xywh'

# ============ 이 아래는 수정 불필요 ============
import os

def _need(*names):
    """이전 셀에서 만들어졌어야 할 변수가 없으면 한국어로 안내하고 False 반환."""
    g = globals()
    missing = [n for n in names if n not in g or g[n] is None]
    if missing:
        print('[중단] 아직 준비되지 않은 값이 있습니다:', ', '.join(missing))
        print('       위쪽 셀을 순서대로 먼저 실행한 뒤 이 셀을 다시 실행하세요.')
        return False
    return True

assert abs(sum(SPLIT) - 1.0) < 1e-6, 'SPLIT 합이 1이 아닙니다.'

_missing_fk = [c for c in CLASS_NAMES if c not in FILE_KEYS_BY_CLASS]
if _missing_fk:
    print('[경고] FILE_KEYS_BY_CLASS에 CLASS_NAMES 항목이 빠져 있습니다:', _missing_fk)
    print('       해당 클래스는 3번 섹션에서 다운로드를 건너뜁니다. CONFIG를 확인하세요.')

print('CONFIG 로드 완료')
print('  DATASET_KEY:', DATASET_KEY)
print('  ACCUM_DIR  :', ACCUM_DIR)
print('  OUT_DIR    :', OUT_DIR)
print('  CLASS_NAMES:', CLASS_NAMES)
print('  PER_CLASS  :', PER_CLASS, '| SPLIT:', SPLIT, '| SEED:', SEED)


## 1. 환경 준비 — Drive 마운트 · 패키지 · 용량 확인

<b>용량 주의:</b> 클래스별로 순차 다운로드하지만, 그래도 다운로드·압축해제·변환본이 동시에 존재하는
동안에는 그 클래스 원본 대비 **2~3배 여유 공간**이 필요합니다(가장 큰 클래스인 "정상"은 최대 76GB).
3번 섹션이 클래스마다 다운로드 전/후 여유공간을 확인하고, 부족하면 경고합니다.

In [ ]:
# Google Drive 마운트 + 출력 폴더 생성 + 패키지 설치 + 용량 확인
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('[안내] Colab 환경이 아니거나 이미 마운트됨:', e)

import os
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(ACCUM_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)
print('출력 폴더 준비:', OUT_DIR)
print('누적 저장 폴더 준비:', ACCUM_DIR)

!pip install -q pyyaml tqdm pillow matplotlib

print('\n=== 디스크 여유 공간 (원본의 2~3배 필요) ===')
!df -h /content /content/drive 2>/dev/null || df -h


## 2. aihubshell 설치 + API 키 입력 + datasetkey 찾기

`aihubshell`을 내려받고, **API 키는 `getpass`로 입력**받아 화면/로그에 남기지 않습니다.
그다음 `-mode l`로 딸기 데이터셋의 **정확한 datasetkey**를 찾아 위 CONFIG의 `DATASET_KEY`에 채웁니다.

In [ ]:
# aihubshell 설치 + API 키 입력(마스킹)
import os, subprocess
from getpass import getpass

AIHUBSHELL = '/content/aihubshell'
try:
    if not os.path.exists(AIHUBSHELL):
        !curl -s -o /content/aihubshell https://api.aihub.or.kr/api/aihubshell.do
        !chmod +x /content/aihubshell
        print('aihubshell 설치 완료:', AIHUBSHELL)
    else:
        print('aihubshell 이미 설치됨(스킵):', AIHUBSHELL)
except Exception as e:
    print('[오류] aihubshell 설치 실패:', e)
    print('       네트워크를 확인하거나 셀을 다시 실행하세요.')

# API 키 입력 — 절대 print/저장하지 않습니다. 셀을 재실행하면 다시 물어봅니다.
AIHUB_API_KEY = getpass('AI Hub API Key 입력(화면에 표시되지 않음): ')
print('API 키 입력 완료 (길이 %d자, 값은 표시하지 않음)' % len(AIHUB_API_KEY))


In [ ]:
# datasetkey 찾기 — 목록에서 "딸기" 항목의 번호를 확인해 CONFIG의 DATASET_KEY에 넣으세요.
try:
    res = subprocess.run(
        [AIHUBSHELL, '-mode', 'l', '-aihubapikey', AIHUB_API_KEY],
        capture_output=True, text=True, timeout=120,
    )
    out = (res.stdout or '') + (res.stderr or '')
    # 키가 로그로 새지 않도록 혹시 모를 노출을 마스킹
    out = out.replace(AIHUB_API_KEY, '****')
    lines = [ln for ln in out.splitlines() if '딸기' in ln or 'strawberr' in ln.lower()]
    if lines:
        print('=== "딸기" 관련 데이터셋 (왼쪽 숫자가 datasetkey) ===')
        print('\n'.join(lines))
    else:
        print('목록에서 "딸기"를 못 찾았습니다. 전체 목록 일부를 출력합니다(키/승인 상태 확인):')
        print('\n'.join(out.splitlines()[:40]))
    print('\n→ 위에서 딸기 데이터셋의 datasetkey를 확인해 CONFIG의 DATASET_KEY에 기입 후 다시 실행하세요.')
except subprocess.TimeoutExpired:
    print('[오류] 목록 조회 시간 초과. 잠시 후 다시 시도하세요.')
except Exception as e:
    print('[오류] 목록 조회 실패:', e)
    print('       API 키/데이터 활용 승인 상태를 확인하세요(승인 전에는 조회가 제한될 수 있음).')


In [ ]:
# (선택) 파일키 목록 확인 — CONFIG의 FILE_KEYS_BY_CLASS가 실제와 맞는지 검증하고 싶을 때만 실행하세요.
# (기본값에 이미 datasetkey=71451의 클래스별 ts/tl filekey가 채워져 있어, 평소에는 이 셀을 건너뛰어도 됩니다.)
if not DATASET_KEY:
    print('[안내] CONFIG의 DATASET_KEY가 비어 있습니다. 위 셀에서 datasetkey를 먼저 확인해 채우세요.')
else:
    try:
        res = subprocess.run(
            [AIHUBSHELL, '-mode', 'l', '-datasetkey', str(DATASET_KEY), '-aihubapikey', AIHUB_API_KEY],
            capture_output=True, text=True, timeout=120,
        )
        out = ((res.stdout or '') + (res.stderr or '')).replace(AIHUB_API_KEY, '****')
        print('=== datasetkey=%s 의 파일키 목록 ===' % DATASET_KEY)
        print(out[:6000])
        print('\n→ 위 목록의 filekey가 CONFIG의 FILE_KEYS_BY_CLASS 값과 다르면 CONFIG를 갱신하세요.')
    except Exception as e:
        print('[오류] 파일키 조회 실패:', e)


## 3. 클래스별 순차 다운로드 → 압축 해제 → 변환 → 샘플링 → 누적 저장 (디스크 절약)

"정상" 클래스 원천만 75.97GB라, 5개 클래스를 한꺼번에 받으면 Colab 디스크가 쉽게 찹니다.
그래서 **클래스 하나씩** 아래 순서를 반복합니다.

```
RAW_DIR 비우기 → 여유공간 확인 → 그 클래스의 ts/tl filekey만 다운로드
  → 압축 해제(분할 *.part* 병합 → zip/tar 해제 → 아카이브 삭제, 중첩 zip 대비 반복)
  → RAW_DIR 스캔·(처음 한 번만) JSON 구조 미리보기 → YOLO txt 변환
  → PER_CLASS(=1,000)장 무작위 샘플 → ACCUM_DIR/{클래스명}/ 에 영구 복사
  → RAW_DIR 전체 삭제(디스크 회수)
```

한 번에 디스크에 남는 원본은 **가장 큰 zip 1개(최대 76GB) 수준**으로 제한됩니다.
`ACCUM_DIR/{클래스명}/images`에 이미 `PER_CLASS`장 이상이 있으면 그 클래스는 **다운로드를 건너뜁니다**
(재실행/이어하기가 가능합니다 — 세션이 끊겨도 처음부터 다시 하지 않아도 됩니다).

aihubshell은 실패(미승인·잘못된 키·인증 오류)에도 종료코드 0으로 끝날 수 있어, 아래 루프는
**aihubshell 출력(API 키 마스킹) 표시 + 실패 문구 감지 + 해제 후 실제 파일 수 검증**으로 이중 확인합니다.
실패한 클래스는 건너뛰고 루프 끝에 **[실패 요약]** 으로 모아 보여주며, 전 클래스 누적이 0장이면
이후 셀이 의미 없으므로 그 자리에서 중단합니다.

처음 다운로드되는 클래스에서 샘플 JSON 1개를 출력합니다. 그 구조를 보고 필요하면
**CONFIG의 `AIHUB_JSON_KEYS` / `AIHUB_BBOX_ORDER`를 조정한 뒤 아래 두 셀(헬퍼 함수, 메인 루프)을 다시 실행**하세요.

In [ ]:
# 클래스별 처리에 쓰는 헬퍼 함수 모음 (스캔/변환/압축해제/디스크 확인) — 기존 4·5번 섹션 로직을 그대로 재사용
import glob, shutil, re, random, json as _json
import zipfile, tarfile
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def find_images(root):
    return sorted(p for p in Path(root).rglob('*') if p.suffix.lower() in IMG_EXTS)

def find_jsons(root):
    return sorted(Path(root).rglob('*.json'))

def _count_media(root):
    n = 0
    for ext in ('jpg', 'jpeg', 'png', 'json', 'zip', 'tar'):
        n += len(glob.glob(os.path.join(root, '**', '*.' + ext), recursive=True))
    return n

def disk_free_gb(path='/content'):
    _total, _used, free = shutil.disk_usage(path)
    return free / (1024 ** 3)

# ---------- 압축 해제 헬퍼 (AI Hub zip은 다운로드만으로는 이미지가 안 나옴 → 반드시 해제 필요) ----------

def _merge_part_files(root):
    """aihubshell 버전에 따라 병합되지 않고 남는 분할 파일(*.zip.part0, *.part1 ...)을
    파일명 순서대로 이어붙여 원본 아카이브로 복원. 병합한 그룹 수를 반환."""
    part_re = re.compile(r'^(?P<base>.+?)\.part(?P<num>\d+)$', re.IGNORECASE)
    groups = defaultdict(list)
    for p in Path(root).rglob('*'):
        if p.is_file():
            m = part_re.match(p.name)
            if m:
                groups[p.parent / m.group('base')].append((int(m.group('num')), p))
    for base, parts in sorted(groups.items()):
        parts.sort(key=lambda t: t[0])
        total_gb = sum(p.stat().st_size for _n, p in parts) / (1024 ** 3)
        if disk_free_gb('/content') < total_gb:
            raise RuntimeError(
                '여유공간 부족: "%s" 분할 병합에 약 %.1fGB가 추가로 필요합니다. '
                'Colab 디스크를 정리한 뒤 다시 실행하세요.' % (base.name, total_gb))
        print('  분할 파일 병합: %s (%d조각, %.1fGB)' % (base.name, len(parts), total_gb))
        with open(base, 'wb') as out_f:
            for _num, p in parts:
                with open(p, 'rb') as in_f:
                    shutil.copyfileobj(in_f, out_f, 64 * 1024 * 1024)
        for _num, p in parts:
            p.unlink()   # 병합 즉시 조각 삭제(디스크 회수)
    return len(groups)

def _fix_zip_name(name):
    """zip 내부 한글 파일명 복원: UTF-8 플래그 없는 zip은 cp437로 잘못 디코딩되므로
    cp949/euc-kr로 재해석을 시도하고, 모두 실패하면 원래 이름 그대로 사용."""
    try:
        raw = name.encode('cp437')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return name
    for enc in ('cp949', 'euc-kr', 'utf-8'):
        try:
            return raw.decode(enc)
        except (UnicodeDecodeError, LookupError):
            continue
    return name

def _extract_zip(arc, dest):
    """zip 1개를 dest 폴더에 해제(한글 파일명 복원 + 경로 이탈 방지). 해제 파일 수 반환."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    n = 0
    with zipfile.ZipFile(arc) as zf:
        for info in zf.infolist():
            name = info.filename
            if not (info.flag_bits & 0x800):      # UTF-8 플래그가 없으면 cp437 오인코딩 복원 시도
                name = _fix_zip_name(name)
            target = (dest / name)
            try:
                target.resolve().relative_to(dest.resolve())
            except ValueError:
                continue                           # zip 내부 경로가 dest 밖을 가리키면 건너뜀
            if info.is_dir() or name.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst, 16 * 1024 * 1024)
            n += 1
            if n % 500 == 0:
                print('    ... %d개 파일 해제 중 (%s)' % (n, arc.name))
    return n

def _extract_tar(arc, dest):
    """tar 1개를 dest 폴더에 해제. 해제 파일 수 반환."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(arc) as tf:
        try:
            tf.extractall(dest, filter='data')     # Python 3.12+ 안전 필터
        except TypeError:
            tf.extractall(dest)
        return sum(1 for m in tf.getmembers() if m.isfile())

def extract_all_archives(root, max_rounds=3):
    """root 하위의 모든 아카이브를 해제한다.
    ① *.part* 분할 파일을 먼저 병합해 원본 zip 복원
    ② .zip은 zipfile(한글 파일명 복원), .tar는 tarfile로 해제
    ③ 해제 성공한 아카이브는 즉시 삭제(디스크 회수)
    ④ 중첩 아카이브(zip 안의 zip) 대비 새 아카이브가 안 나올 때까지 최대 max_rounds회 반복
    여유공간이 아카이브 크기보다 적으면 RuntimeError를 던진다(해제본이 그만큼 추가로 필요)."""
    merged = _merge_part_files(root)
    if merged:
        print('  분할 파일 그룹 %d개 병합 완료' % merged)
    total = 0
    for round_no in range(1, max_rounds + 1):
        archives = sorted(p for p in Path(root).rglob('*')
                          if p.is_file() and p.suffix.lower() in ('.zip', '.tar'))
        if not archives:
            break
        print('  [압축 해제 %d회차] 아카이브 %d개 발견' % (round_no, len(archives)))
        for arc in archives:
            arc_gb = arc.stat().st_size / (1024 ** 3)
            free = disk_free_gb('/content')
            if free < arc_gb:
                raise RuntimeError(
                    '여유공간 부족: 남은 공간 %.1fGB, "%s" 해제에 약 %.1fGB가 추가로 필요합니다. '
                    'Colab 디스크를 정리하거나 런타임을 재시작한 뒤 다시 실행하세요.'
                    % (free, arc.name, arc_gb))
            print('  해제 중: %s (%.1fGB, 여유 %.1fGB)' % (arc.name, arc_gb, free))
            dest = arc.parent / arc.stem           # 아카이브별 폴더에 해제(이름 충돌 방지)
            try:
                if arc.suffix.lower() == '.zip':
                    n = _extract_zip(arc, dest)
                else:
                    n = _extract_tar(arc, dest)
            except RuntimeError:
                raise
            except Exception as e:
                print('  [경고] "%s" 해제 실패(건너뜀): %s' % (arc.name, e))
                arc.rename(arc.with_name(arc.name + '.failed'))   # 다음 회차에 다시 안 걸리게 제외
                continue
            arc.unlink()                            # 해제 완료 → 아카이브 삭제(디스크 회수)
            total += 1
            print('    → %d개 파일 해제 완료, 아카이브 삭제(디스크 회수)' % n)
    return total

# ---------- 클래스/좌표 변환 헬퍼 ----------

def resolve_class_index(raw, img_path):
    """원천 클래스 표기 → CLASS_NAMES 인덱스. 매핑 실패 시 None."""
    if raw is None:
        # 최후의 수단: 상위 폴더명에서 클래스 추정(폴더로 나뉜 데이터 대비)
        for part in reversed(img_path.parts):
            name = KOR_TO_MODEL.get(part, part)
            if name in CLASS_NAMES:
                return CLASS_NAMES.index(name)
        return None
    name = KOR_TO_MODEL.get(str(raw), str(raw))
    if name in CLASS_NAMES:
        return CLASS_NAMES.index(name)
    return None

def parse_bbox(bb):
    """다양한 bbox 표기 → (x1,y1,x2,y2) 픽셀."""
    try:
        if AIHUB_BBOX_ORDER == 'xyxy_dict':
            for ks in (('xtl', 'ytl', 'xbr', 'ybr'), ('x1', 'y1', 'x2', 'y2')):
                if isinstance(bb, dict) and all(k in bb for k in ks):
                    return [float(bb[k]) for k in ks]
            return None
        vals = [float(v) for v in bb]
        if AIHUB_BBOX_ORDER == 'xyxy':
            return vals[:4]
        if AIHUB_BBOX_ORDER == 'xywh':
            x, y, w, h = vals[:4]
            return [x, y, x + w, y + h]
    except Exception:
        return None
    return None

def convert_raw_to_yolo(raw_images, raw_jsons, conv_dir, show_sample=False):
    """RAW_DIR에서 찾은 이미지/JSON을 YOLO txt로 변환해 conv_dir에 쓰고 records 리스트 반환.
    (기존 5번 섹션 변환 로직과 동일 — 클래스별 루프에서 매 클래스마다 재사용)"""
    conv_dir = Path(conv_dir)
    if conv_dir.exists():
        shutil.rmtree(conv_dir)
    conv_dir.mkdir(parents=True)

    K = AIHUB_JSON_KEYS
    json_by_stem = {p.stem: p for p in raw_jsons}

    if show_sample and raw_jsons:
        sample = raw_jsons[0]
        try:
            data = _json.load(open(sample, encoding='utf-8'))
            pretty = _json.dumps(data, ensure_ascii=False, indent=2)
            print('=== 샘플 JSON:', sample, '===')
            print(pretty[:2000])
            if len(pretty) > 2000:
                print('... (생략) ...')
            print('최상위 키:', list(data.keys()) if isinstance(data, dict) else type(data))
            print('→ 구조가 다르면 CONFIG의 AIHUB_JSON_KEYS / AIHUB_BBOX_ORDER 를 조정한 뒤 이 두 셀을 다시 실행하세요.\n')
        except Exception as e:
            print('[오류] 샘플 JSON 미리보기 실패:', e)

    records = []                 # {'img','txt','cls','stem'}
    stats = Counter()            # ok / no_json / parse_err / no_box / bg
    unmapped = Counter()         # 매핑 실패 클래스 표기 → 개수
    key_warned = False

    for img in raw_images:
        jp = json_by_stem.get(img.stem)
        if jp is None:
            cid = resolve_class_index(None, img)
            if ALLOW_BACKGROUND and cid is not None:
                (conv_dir / (img.stem + '.txt')).write_text('', encoding='utf-8')
                records.append({'img': img, 'txt': conv_dir / (img.stem + '.txt'), 'cls': cid, 'stem': img.stem})
                stats['bg'] += 1
            else:
                stats['no_json'] += 1
            continue
        try:
            data = _json.load(open(jp, encoding='utf-8'))
        except Exception:
            stats['parse_err'] += 1
            continue

        anns = data.get(K['annotations']) if isinstance(data, dict) else None
        if anns is None:
            if not key_warned:
                key_warned = True
                print("[오류] JSON에 '%s' 키가 없습니다. 실제 최상위 키: %s"
                      % (K['annotations'], list(data.keys()) if isinstance(data, dict) else '?'))
                print('       CONFIG의 AIHUB_JSON_KEYS를 실제 키로 고치고 다시 실행하세요.')
            stats['parse_err'] += 1
            continue
        if isinstance(anns, dict):
            anns = [anns]

        W = data.get(K['img_width']); H = data.get(K['img_height'])
        if not W or not H:
            try:
                with Image.open(img) as im:
                    W, H = im.size
            except Exception:
                stats['parse_err'] += 1
                continue
        W, H = float(W), float(H)

        lines, box_cls = [], []
        for ann in anns:
            if not isinstance(ann, dict):
                continue
            raw_cls = ann.get(K['class'], data.get(K['class']) if isinstance(data, dict) else None)
            bb = ann.get(K['bbox'])
            if bb is None:
                continue
            cid = resolve_class_index(raw_cls, img)
            if cid is None:
                unmapped[str(raw_cls)] += 1
                continue
            xyxy = parse_bbox(bb)
            if xyxy is None:
                continue
            x1, y1, x2, y2 = xyxy
            x1, x2 = max(0.0, min(x1, x2)), min(W, max(x1, x2))
            y1, y2 = max(0.0, min(y1, y2)), min(H, max(y1, y2))
            if x2 - x1 < 1 or y2 - y1 < 1:
                continue
            cx = min(max((x1 + x2) / 2 / W, 0.0), 1.0)
            cy = min(max((y1 + y2) / 2 / H, 0.0), 1.0)
            bw = min((x2 - x1) / W, 1.0)
            bh = min((y2 - y1) / H, 1.0)
            lines.append('%d %.6f %.6f %.6f %.6f' % (cid, cx, cy, bw, bh))
            box_cls.append(cid)

        out = conv_dir / (img.stem + '.txt')
        if lines:
            out.write_text('\n'.join(lines), encoding='utf-8')
            dom = sorted(Counter(box_cls).items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
            records.append({'img': img, 'txt': out, 'cls': dom, 'stem': img.stem})
            stats['ok'] += 1
        else:
            # 박스 0개 → 배경 처리(클래스가 추정되면)
            cid = resolve_class_index(None, img)
            if ALLOW_BACKGROUND and cid is not None:
                out.write_text('', encoding='utf-8')
                records.append({'img': img, 'txt': out, 'cls': cid, 'stem': img.stem})
                stats['bg'] += 1
            else:
                stats['no_box'] += 1

    print('  변환 통계: 성공(박스 있음)=%d, 배경=%d, JSON없음=%d, 파싱오류=%d, 박스0/미채택=%d, 레코드총=%d'
          % (stats['ok'], stats['bg'], stats['no_json'], stats['parse_err'], stats['no_box'], len(records)))
    if unmapped:
        print('  [경고] CLASS_NAMES/KOR_TO_MODEL로 매핑 안 된 클래스 표기(건너뜀):', dict(unmapped.most_common(10)))
        print('         → CONFIG의 KOR_TO_MODEL 또는 CLASS_NAMES를 고쳐 이 두 셀을 다시 실행하세요.')
    return records


In [ ]:
# 클래스별 순차 처리: 다운로드 → 압축 해제 → 변환 → PER_CLASS 샘플 → ACCUM_DIR 누적 → RAW_DIR 삭제
CONV_DIR = '/content/converted_labels'
rng = random.Random(SEED)   # 클래스 루프 밖에서 한 번만 생성해 재사용 → 클래스마다 다른 셔플이 되지 않고 재현성 유지
_shown_sample = False
accum_before, accum_after = {}, {}
failed_classes = []          # (클래스명, 실패 사유) — 루프 끝에 요약 출력

# aihubshell은 실패(미승인/잘못된 키/인증 오류)에도 종료코드 0으로 끝나는 경우가 있어,
# 출력 문자열과 실제 다운로드 결과(파일 수)로 이중 검증합니다.
_FAIL_MARKERS = ['페이지가 존재하지 않습니다', '승인', '권한', 'Unauthorized', 'Error', '실패']

def _fail_guidance(class_name):
    print('[실패] "%s" 처리에 실패했습니다. 원인 후보를 순서대로 확인하세요:' % class_name)
    print('  ① AI Hub 마이페이지에서 이 데이터셋(웹 dataSetSn 71451)의 "다운로드 신청"이 승인됐는지 확인')
    print('     (승인은 수동 심사라 수 시간~수일 걸릴 수 있습니다. 승인 전에는 다운로드가 조용히 실패합니다)')
    print('  ② 0번 CONFIG 셀을 다시 실행해 DATASET_KEY=%r 값이 메모리에 반영됐는지 확인' % DATASET_KEY)
    print('  ③ API 키가 유효한지 확인 (2번 섹션 셀을 재실행해 키를 다시 입력)')

for class_name in CLASS_NAMES:
    print('\n' + '=' * 60)
    print('[클래스]', class_name)
    print('=' * 60)

    acc_img_dir = Path(ACCUM_DIR) / class_name / 'images'
    acc_lbl_dir = Path(ACCUM_DIR) / class_name / 'labels'
    acc_img_dir.mkdir(parents=True, exist_ok=True)
    acc_lbl_dir.mkdir(parents=True, exist_ok=True)

    already = len(list(acc_img_dir.glob('*')))
    accum_before[class_name] = already
    if already >= PER_CLASS:
        print('[스킵] ACCUM_DIR에 이미 %d장 확보됨(목표 %d) → 이 클래스는 다운로드를 건너뜁니다.' % (already, PER_CLASS))
        accum_after[class_name] = already
        continue

    fk = FILE_KEYS_BY_CLASS.get(class_name)
    if not fk:
        print('[실패] FILE_KEYS_BY_CLASS에 "%s" 항목이 없습니다. CONFIG를 확인하세요.' % class_name)
        failed_classes.append((class_name, 'FILE_KEYS_BY_CLASS에 항목 없음(CONFIG 확인)'))
        accum_after[class_name] = already
        continue

    # 1) RAW_DIR 비우기(이전 클래스 잔여물 삭제)
    if os.path.exists(RAW_DIR):
        shutil.rmtree(RAW_DIR)
    os.makedirs(RAW_DIR, exist_ok=True)

    # 2) 다운로드 전 여유공간 확인 (안전장치: 예상 zip 크기의 1.5배 미만이면 경고, 아예 못 받을 정도면 건너뜀)
    expected_gb = CLASS_TS_SIZE_GB.get(class_name)
    free_before = disk_free_gb('/content')
    print('다운로드 전 여유공간: %.1fGB (이 클래스 예상 용량 약 %s GB)'
          % (free_before, ('%.1f' % expected_gb) if expected_gb else '?'))
    if expected_gb:
        if free_before < expected_gb:
            print('[실패] 여유공간(%.1fGB)이 예상 용량(%.1fGB)보다 적어 다운로드를 받을 수 없습니다.' % (free_before, expected_gb))
            print('       Colab 디스크를 정리하거나 런타임을 재시작한 뒤 다시 실행하세요. 이 클래스는 건너뜁니다.')
            failed_classes.append((class_name, '디스크 여유공간 부족(%.1fGB < %.1fGB)' % (free_before, expected_gb)))
            accum_after[class_name] = already
            continue
        if free_before < expected_gb * 1.5:
            print('[경고] 여유공간(%.1fGB)이 안전 마진(예상 용량의 1.5배 = %.1fGB) 미만입니다. 계속 진행하지만 주의하세요.'
                  % (free_before, expected_gb * 1.5))

    # 3) 다운로드 (원천ts + 라벨tl 만) — 출력을 캡처해 실패 문자열을 감지하고, API 키는 마스킹해 표시
    filekey_arg = '%s,%s' % (fk['ts'], fk['tl'])
    cmd = [AIHUBSHELL, '-mode', 'd', '-datasetkey', str(DATASET_KEY),
           '-filekey', filekey_arg, '-aihubapikey', AIHUB_API_KEY]
    print('다운로드: datasetkey=%s filekey=%s (원천+라벨만)' % (DATASET_KEY, filekey_arg))
    try:
        res = subprocess.run(cmd, cwd=RAW_DIR, capture_output=True, text=True, timeout=60 * 60 * 6)
        dl_out = (res.stdout or '') + '\n' + (res.stderr or '')
        if AIHUB_API_KEY:
            dl_out = dl_out.replace(AIHUB_API_KEY, '****')   # 키가 로그로 새지 않도록 마스킹
        tail = dl_out.strip().splitlines()[-30:]
        print('--- aihubshell 출력 (마지막 %d줄, API 키 마스킹) ---' % len(tail))
        print('\n'.join(tail))
        print('--- 출력 끝 (종료 코드: %s) ---' % res.returncode)
        bad = [m for m in _FAIL_MARKERS if m in dl_out]
        if res.returncode != 0 or bad:
            reason = ('종료 코드 %d' % res.returncode) if res.returncode != 0 \
                     else '출력에서 실패 문구 감지: %s' % ', '.join(bad)
            print('[실패] 다운로드가 실패한 것으로 판단됩니다 (%s).' % reason)
            _fail_guidance(class_name)
            failed_classes.append((class_name, '다운로드 실패(%s)' % reason))
            accum_after[class_name] = already
            continue
    except subprocess.TimeoutExpired:
        print('[실패] 다운로드 시간 초과 →', class_name, '건너뜀')
        failed_classes.append((class_name, '다운로드 시간 초과'))
        accum_after[class_name] = already
        continue
    except Exception as e:
        print('[실패] 다운로드 실패:', e, '→', class_name, '건너뜀')
        failed_classes.append((class_name, '다운로드 예외: %s' % e))
        accum_after[class_name] = already
        continue

    free_after_dl = disk_free_gb('/content')
    print('다운로드 후 여유공간: %.1fGB' % free_after_dl)

    # 4) 압축 해제 — AI Hub 파일은 zip(TS_*.zip/TL_*.zip)으로 오므로 반드시 해제해야 이미지가 나옵니다.
    #    분할 파일(*.part*) 병합 → zip/tar 해제 → 중첩 아카이브 대비 반복 (헬퍼 셀의 extract_all_archives)
    try:
        n_arc = extract_all_archives(RAW_DIR)
        print('압축 해제 완료: 아카이브 %d개 처리, 해제 후 여유공간 %.1fGB' % (n_arc, disk_free_gb('/content')))
    except Exception as e:
        print('[실패] "%s" 압축 해제 중 오류: %s' % (class_name, e))
        failed_classes.append((class_name, '압축 해제 실패: %s' % e))
        accum_after[class_name] = already
        continue

    # 5) 실질 검증 — 다운로드+해제 후 미디어 파일(이미지+JSON+아카이브)이 하나도 없으면 무조건 실패 처리.
    #    (aihubshell은 미승인/오류에도 종료코드 0을 반환할 수 있어 이 검증이 최종 안전장치입니다)
    media_n = _count_media(RAW_DIR)
    if media_n == 0:
        print('[실패] 다운로드·해제 결과 RAW_DIR에 파일이 하나도 없습니다.')
        _fail_guidance(class_name)
        failed_classes.append((class_name, '다운로드 결과 파일 0개(승인/DATASET_KEY/API 키 확인)'))
        accum_after[class_name] = already
        continue

    # 6) RAW_DIR 스캔
    raw_images = find_images(RAW_DIR)
    raw_jsons = find_jsons(RAW_DIR)
    print('이미지 %d장, JSON %d개 발견' % (len(raw_images), len(raw_jsons)))
    if not raw_images:
        print('[실패] 해제 후에도 이미지가 없습니다(미디어 파일 %d개는 있음).' % media_n)
        _fail_guidance(class_name)
        failed_classes.append((class_name, '해제 후 이미지 0장'))
        accum_after[class_name] = already
        continue

    # 7) YOLO 변환 (처음 다운로드되는 클래스에서만 샘플 JSON을 출력해 키 매핑 확인을 유도)
    records = convert_raw_to_yolo(raw_images, raw_jsons, CONV_DIR, show_sample=not _shown_sample)
    _shown_sample = True

    # 이 클래스로 받은 데이터 중 실제로 이 클래스로 매핑된 레코드만 채택(다른 클래스로 잘못 매핑된 것은 제외)
    target_cid = CLASS_NAMES.index(class_name)
    class_records = [r for r in records if r['cls'] == target_cid]
    other = len(records) - len(class_records)
    if other:
        print('[참고] 이 클래스 다운로드분 중 다른 클래스로 매핑된 레코드 %d개는 제외합니다.' % other)

    # 8) PER_CLASS 무작위 샘플(클래스 루프 밖에서 만든 rng를 그대로 재사용 → 재현성)
    rng.shuffle(class_records)
    need = max(0, PER_CLASS - already)
    take = class_records[:need]
    print('발견 %d장 → 신규 채택 %d장 (기존 누적 %d + 신규 %d = %d / 목표 %d)'
          % (len(class_records), len(take), already, len(take), already + len(take), PER_CLASS))
    if len(class_records) < need:
        print('[주의] "%s": 목표 %d장에 %d장 부족 → 있는 만큼만 사용합니다.' % (class_name, PER_CLASS, need - len(class_records)))

    # 9) ACCUM_DIR로 영구 복사
    for r in take:
        img, txt = r['img'], r['txt']
        dst_img = acc_img_dir / img.name
        k = 1
        while dst_img.exists():
            dst_img = acc_img_dir / ('%s_%d%s' % (img.stem, k, img.suffix.lower()))
            k += 1
        shutil.copy2(img, dst_img)
        dst_txt = acc_lbl_dir / (dst_img.stem + '.txt')
        if txt is not None and os.path.exists(txt):
            shutil.copy2(txt, dst_txt)
        else:
            dst_txt.write_text('', encoding='utf-8')

    accum_after[class_name] = len(list(acc_img_dir.glob('*')))

    # 10) RAW_DIR 삭제 → 디스크 회수
    shutil.rmtree(RAW_DIR, ignore_errors=True)
    os.makedirs(RAW_DIR, exist_ok=True)
    free_after_cleanup = disk_free_gb('/content')
    print('RAW_DIR 삭제 후 여유공간: %.1fGB (회수 약 %.1fGB)' % (free_after_cleanup, free_after_cleanup - free_after_dl))

print('\n' + '=' * 60)
print('클래스별 처리 완료. ACCUM_DIR:', ACCUM_DIR)

# ---- 실패 클래스 요약 (있으면 크게 표시) ----
if failed_classes:
    print('\n' + '!' * 60)
    print('[실패 요약] 다음 %d개 클래스는 처리에 실패했습니다:' % len(failed_classes))
    for _c, _reason in failed_classes:
        print('  - %-8s : %s' % (_c, _reason))
    print('위의 해당 클래스 출력에서 [실패]/[경고] 줄과 aihubshell 출력을 확인하세요.')
    print('!' * 60)

# ---- 전 클래스 누적 합계가 0이면 이후 셀이 의미 없으므로 확실히 중단 ----
_total_accum = sum(len(list((Path(ACCUM_DIR) / c / 'images').glob('*'))) for c in CLASS_NAMES)
print('전 클래스 누적 합계: %d장' % _total_accum)
if _total_accum == 0:
    raise SystemExit(
        '[중단] 모든 클래스의 누적 샘플이 0장입니다. 이후 셀(분할/data.yaml)을 실행해도 의미가 없어 여기서 멈춥니다.\n'
        '       위 3번 섹션의 클래스별 처리 출력에서 [실패]/[경고] 줄을 확인하세요.\n'
        '       흔한 원인: ① AI Hub 다운로드 신청 미승인 ② CONFIG 셀 미재실행(DATASET_KEY 미반영) ③ API 키 오류')


In [ ]:
# 클래스별 최종 누적 샘플 수 요약 바차트
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['axes.unicode_minus'] = False
final_counts = {c: len(list((Path(ACCUM_DIR) / c / 'images').glob('*'))) for c in CLASS_NAMES}

print('=== 클래스별 최종 누적 샘플 수 (ACCUM_DIR 기준) ===')
for c in CLASS_NAMES:
    before = accum_before.get(c, 0)
    print('  %-8s : %6d → %5d 장 (목표 %d)' % (c, before, final_counts[c], PER_CLASS))
print('  샘플 총합:', sum(final_counts.values()))

idx = list(range(len(CLASS_NAMES)))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(idx, [final_counts[c] for c in CLASS_NAMES])
ax.axhline(PER_CLASS, color='red', linestyle='--', linewidth=1, label='목표(%d)' % PER_CLASS)
ax.set_xticks(idx); ax.set_xticklabels(CLASS_NAMES)
ax.set_title('ACCUM_DIR 클래스별 누적 샘플 수'); ax.legend()
plt.tight_layout(); plt.show()


## 4. ACCUM_DIR → sampled 리스트 재구성

이제부터는 `RAW_DIR`를 다시 스캔하지 않고, 위 3번 섹션에서 클래스별로 이미 선별해 `ACCUM_DIR`에
저장해 둔 이미지·라벨만 읽어 `sampled` 리스트를 만듭니다. 이 아래 분할(5번)·data.yaml(6번) 로직은
기존과 동일하게 `sampled`를 그대로 사용합니다.

In [ ]:
# ACCUM_DIR에 누적된 클래스별 이미지+라벨을 읽어 sampled 리스트로 재구성
sampled = []
counts = {}
for cid, class_name in enumerate(CLASS_NAMES):
    img_dir = Path(ACCUM_DIR) / class_name / 'images'
    lbl_dir = Path(ACCUM_DIR) / class_name / 'labels'
    imgs = find_images(img_dir) if img_dir.exists() else []
    for img in imgs:
        txt = lbl_dir / (img.stem + '.txt')
        sampled.append({'img': img, 'txt': txt if txt.exists() else None, 'cls': cid, 'stem': img.stem})
    counts[class_name] = len(imgs)

print('=== ACCUM_DIR 기준 클래스별 샘플 수 ===')
for c in CLASS_NAMES:
    print('  %-8s : %5d장' % (c, counts[c]))
print('  샘플 총합:', len(sampled))

if not sampled:
    raise SystemExit(
        '[중단] ACCUM_DIR(%s)에서 이미지를 하나도 찾지 못했습니다. 이대로는 분할/데이터셋 생성이 불가능합니다.\n'
        '       3번 섹션의 클래스별 처리 출력에서 [실패]/[경고] 줄을 확인해 원인(승인/DATASET_KEY/API 키)을\n'
        '       해결한 뒤, 3번 섹션부터 다시 실행하세요.' % ACCUM_DIR)


## 5. train / val / test 층화 분할 (8:1:1)

클래스 균형을 유지하며 `SEED` 고정으로 분할합니다.
<b>누수 방지:</b> 같은 개체/연속 프레임이 여러 split에 흩어지지 않도록, 파일명에서 **개체ID(그룹키)**를 뽑아
**그룹 단위**로 나눕니다. 개체ID를 식별할 수 없으면 이미지 단위로 나누고 그 사실을 경고로 남깁니다.
최종 폴더는 학습 노트북이 기대하는 `images/{train,val,test}` · `labels/{train,val,test}` 구조로 `OUT_DIR`에 만듭니다.

In [ ]:
# 그룹 인식 층화 분할 → OUT_DIR 로 복사
if not globals().get('sampled'):
    raise SystemExit(
        '[중단] sampled 리스트가 비어 있어 train/val/test 분할을 진행할 수 없습니다.\n'
        '       4번 섹션(ACCUM_DIR → sampled 재구성) 셀을 먼저 실행하세요. 그래도 비어 있다면\n'
        '       3번 섹션의 클래스별 처리 출력에서 [실패]/[경고] 줄을 확인해 원인을 해결한 뒤 다시 실행하세요.')

def group_key(stem):
    if GROUP_ID_REGEX:
        m = re.search(GROUP_ID_REGEX, stem)
        if m:
            return m.group(1) if m.groups() else m.group(0)
    # 자동 추정: 끝의 _0001 / -12 / 프레임번호 제거
    return re.sub(r'[_\-]?\d+$', '', stem)

# 개체ID가 의미 있는지 판정(그룹 수가 이미지 수보다 충분히 적으면 그룹 분할)
groups_all = {group_key(r['stem']) for r in sampled}
use_group = len(groups_all) < 0.95 * max(1, len(sampled))
if use_group:
    print('개체ID(그룹) 기준 분할: 그룹 %d개 / 이미지 %d장 (누수 방지)' % (len(groups_all), len(sampled)))
else:
    print('[경고] 파일명에서 개체ID를 식별하기 어려워 이미지 단위로 분할합니다.')
    print('       연속 프레임이 있다면 GROUP_ID_REGEX를 지정해 다시 실행하는 것을 권장합니다.')

random.seed(SEED)
tr_r, va_r, te_r = SPLIT
split_map = {'train': [], 'val': [], 'test': []}

# 클래스별로 그룹을 모아 층화 분할
cls_to_groups = defaultdict(lambda: defaultdict(list))  # cls -> gkey -> [records]
for r in sampled:
    gk = group_key(r['stem']) if use_group else r['stem']
    cls_to_groups[r['cls']][gk].append(r)

for cid, gmap in cls_to_groups.items():
    gkeys = list(gmap.keys())
    random.shuffle(gkeys)
    n = len(gkeys)
    n_val = max(1, round(n * va_r)) if n >= 3 else 0
    n_test = max(1, round(n * te_r)) if n >= 3 else 0
    val_g = gkeys[:n_val]
    test_g = gkeys[n_val:n_val + n_test]
    train_g = gkeys[n_val + n_test:]
    for split, gs in (('val', val_g), ('test', test_g), ('train', train_g)):
        for gk in gs:
            split_map[split].extend(gmap[gk])

# OUT_DIR 폴더 초기화(이미지/라벨 하위만) 후 복사
for s in ('train', 'val', 'test'):
    for sub in ('images', 'labels'):
        d = Path(OUT_DIR) / sub / s
        if d.exists():
            shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)

used = set()
for split, items in split_map.items():
    for r in items:
        stem, k = r['stem'], 1
        while stem in used:
            stem = '%s_%d' % (r['stem'], k); k += 1
        used.add(stem)
        img = r['img']
        shutil.copy2(img, Path(OUT_DIR) / 'images' / split / (stem + img.suffix.lower()))
        dst_lbl = Path(OUT_DIR) / 'labels' / split / (stem + '.txt')
        if r['txt'] is not None and os.path.exists(r['txt']):
            shutil.copy2(r['txt'], dst_lbl)
        else:
            dst_lbl.write_text('', encoding='utf-8')

print('\n=== split별 이미지 수 ===')
for s in ('train', 'val', 'test'):
    print('  %-5s : %d장' % (s, len(split_map[s])))


## 6. data.yaml 생성 + 검증

학습 노트북이 그대로 읽는 형식으로 `data.yaml`을 `OUT_DIR`에 저장합니다
(`path` / `train: images/train` / `val: images/val` / `test: images/test` / `names: {0: ...}`).
그다음 **이미지 수 ↔ 라벨 수 일치**, **클래스 인덱스 범위**를 검증합니다.

In [ ]:
# data.yaml 저장 + 무결성 검증
import yaml

data_yaml_path = str(Path(OUT_DIR) / 'data.yaml')
with open(data_yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump({
        'path': str(OUT_DIR),
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'names': {i: n for i, n in enumerate(CLASS_NAMES)},
    }, f, allow_unicode=True, sort_keys=False)
print('data.yaml 생성:', data_yaml_path)
print(open(data_yaml_path, encoding='utf-8').read())

# 검증
ok_all = True
max_cls = len(CLASS_NAMES) - 1
for s in ('train', 'val', 'test'):
    imgs = find_images(Path(OUT_DIR) / 'images' / s)
    lbls = list((Path(OUT_DIR) / 'labels' / s).glob('*.txt'))
    print('  %-5s: 이미지 %d, 라벨 %d' % (s, len(imgs), len(lbls)))
    if len(imgs) != len(lbls):
        ok_all = False
        print('    [경고] 이미지 수와 라벨 수가 다릅니다.')
    for lp in lbls:
        for ln in open(lp, encoding='utf-8'):
            ln = ln.strip()
            if not ln:
                continue
            c = int(float(ln.split()[0]))
            if c < 0 or c > max_cls:
                ok_all = False
                print('    [경고] 클래스 인덱스 범위 초과: %s (%s)' % (c, lp.name))
                break
print('\n검증 결과:', '통과 ✅' if ok_all else '경고 있음 ⚠️ (위 메시지 확인)')


In [ ]:
# 무작위 5장 bbox 시각화 — 변환 좌표가 맞는지 눈으로 확인
import matplotlib.pyplot as plt
import matplotlib.patches as patches

train_imgs = find_images(Path(OUT_DIR) / 'images' / 'train')
if not train_imgs:
    print('[안내] train 이미지가 없어 시각화를 건너뜁니다.')
else:
    random.seed(SEED)
    picks = random.sample(train_imgs, min(5, len(train_imgs)))
    fig, axes = plt.subplots(1, len(picks), figsize=(4 * len(picks), 4))
    if len(picks) == 1:
        axes = [axes]
    for ax, img in zip(axes, picks):
        try:
            im = Image.open(img).convert('RGB')
            W, H = im.size
            ax.imshow(im); ax.axis('off'); ax.set_title(img.stem[:16], fontsize=8)
            lbl = Path(OUT_DIR) / 'labels' / 'train' / (img.stem + '.txt')
            if lbl.exists():
                for ln in open(lbl, encoding='utf-8'):
                    p = ln.split()
                    if len(p) != 5:
                        continue
                    c, cx, cy, bw, bh = int(float(p[0])), *[float(v) for v in p[1:]]
                    x = (cx - bw / 2) * W; y = (cy - bh / 2) * H
                    ax.add_patch(patches.Rectangle((x, y), bw * W, bh * H,
                                 fill=False, edgecolor='lime', linewidth=2))
                    ax.text(x, max(0, y - 4), str(c), color='lime', fontsize=9)
        except Exception as e:
            ax.set_title('오류: %s' % e, fontsize=7)
    plt.tight_layout(); plt.show()
    print('초록 박스가 병반 위치에 맞으면 변환이 정상입니다(숫자는 클래스 id).')


## 7. 학습 노트북으로 연결

이제 준비가 끝났습니다. `OUT_DIR`(위 `data.yaml`이 있는 폴더)를 학습 노트북의 입력으로 넘깁니다.

학습 노트북 `doctorgreen_yolo_map_boost.ipynb`의 **0번 CONFIG 셀**을 다음처럼 설정하세요.
```python
DATASET_DIR  = '<이 노트북의 OUT_DIR 값>'   # 예: /content/drive/MyDrive/doctor_green_training/dataset
LABEL_FORMAT = 'yolo'                         # 이미 YOLO txt로 변환해 두었으므로 그대로 소비됨
DATA_YAML    = '<OUT_DIR>/data.yaml'          # 이 노트북이 만든 분할을 그대로 쓰려면 지정(권장)
                                              #  '' 로 두면 학습 노트북이 자체 재분할함(그래도 동작)
```

<div style="border:2px solid #d33; background:#fff3f3; padding:12px; border-radius:8px">
<b>⚠️ 배포 전 마지막 확인</b><br>
이 노트북의 <code>CLASS_NAMES</code> 순서 = <code>data.yaml</code>의 names 순서 = 배포 모델 <code>model.names</code> 순서가
<b>모두 같아야</b> 합니다. 학습 노트북 9장(배포)의 클래스 검증 셀에서 "일치"를 확인한 뒤에만 업로드하세요.
현재 CLASS_NAMES: <code>['정상','역병','시들음병','잎끝마름','황화']</code> — 배포 모델이 영문이면 CONFIG에서 바꿨는지 재확인.
</div>
